In [16]:
# Cell 2: Load CSV
df = pd.read_csv("agriculture_dataset.csv")   # change name if different

print("Shape:", df.shape)
df.head()


Shape: (212019, 32)


,High_Resolution_RGB,Multispectral_Images,Thermal_Images,Temporal_Images,Spatial_Resolution,GPS_Coordinates,Field_Boundaries,Elevation_Data,Canopy_Coverage,NDVI,...,Weed_Coverage,Pest_Damage,Crop_Growth_Stage,Expected_Yield,Crop_Type,Ground_Truth_Segmentation,Bounding_Boxes,Water_Flow,Drainage_Features,Crop_Health_Label
0,0,0,0,0,0.667324,201538,3,28.207634,8.046926,0.676945,...,1.922274,84,2,2540.784327,Wheat,1,5,41.771884,0,1
1,1,1,0,0,1.459000,215854,3,82.335147,147.512332,0.414781,...,4.851381,56,3,3227.617025,Wheat,0,1,27.564635,0,1
2,0,0,0,0,0.500442,890802,3,83.865629,30.246527,0.723610,...,5.974859,38,1,4609.938146,Maize,1,8,29.510836,0,1
3,0,0,0,0,1.865161,605584,3,20.747905,6.857820,0.405611,...,2.100598,27,2,1409.716754,Maize,0,1,34.822855,0,0
4,0,1,1,1,1.392331,871732,3,22.588815,26.168558,0.465992,...,3.025669,84,4,3905.312588,Rice,0,2,15.493255,1,0


In [15]:
# Cell 1: Imports
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [20]:
import pandas as pd
import numpy as np

target_col = "Crop_Health_Label"

# Separate X and y
X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

# Encode y if it's text
if y.dtype == "object":
    y = y.astype("category").cat.codes

# ✅ Convert categorical/text columns in X to numeric
cat_cols = X.select_dtypes(include=["object", "category"]).columns
print("Categorical columns in X:", list(cat_cols))

X = pd.get_dummies(X, columns=cat_cols, drop_first=True)


Categorical columns in X: ['Crop_Type']


In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)


In [23]:
print("Any non-numeric left in X?", X.select_dtypes(include=["object","category"]).shape[1] > 0)
print("X shape after encoding:", X.shape)


Any non-numeric left in X? False
X shape after encoding: (212019, 32)


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

baseline = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print("Baseline accuracy:", accuracy_score(y_test, baseline_pred))


Baseline accuracy: 0.5014385435336289


In [25]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.long)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_train_t)
test_ds  = TensorDataset(X_test_t,  y_test_t)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256)


In [26]:
class_counts = np.bincount(y_train.values)
class_weights = class_counts.sum() / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

print("Class counts:", class_counts)
print("Class weights:", class_weights)


Class counts: [ 51057 118558]
Class weights: tensor([1.6610, 0.7153])


In [27]:
import torch.nn as nn

class TabularNet(nn.Module):
    def __init__(self, input_dim, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = TabularNet(input_dim=X_train.shape[1]).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

epochs = 40


Using device: cpu


In [29]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, correct = 0, 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


In [30]:
def evaluate(model, loader):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(1) == yb).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


In [31]:
for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    test_loss, test_acc = evaluate(model, test_loader)

    scheduler.step(test_loss)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}"
    )


Epoch 01 | Train Loss: 0.6981, Train Acc: 0.5020 | Test Loss: 0.6942, Test Acc: 0.4920
Epoch 02 | Train Loss: 0.6946, Train Acc: 0.5080 | Test Loss: 0.6941, Test Acc: 0.4968
Epoch 03 | Train Loss: 0.6943, Train Acc: 0.5085 | Test Loss: 0.6944, Test Acc: 0.4118
Epoch 04 | Train Loss: 0.6939, Train Acc: 0.5090 | Test Loss: 0.6940, Test Acc: 0.5645
Epoch 05 | Train Loss: 0.6937, Train Acc: 0.5112 | Test Loss: 0.6937, Test Acc: 0.5283
Epoch 06 | Train Loss: 0.6935, Train Acc: 0.5157 | Test Loss: 0.6944, Test Acc: 0.5123
Epoch 07 | Train Loss: 0.6932, Train Acc: 0.5144 | Test Loss: 0.6938, Test Acc: 0.5532
Epoch 08 | Train Loss: 0.6928, Train Acc: 0.5142 | Test Loss: 0.6951, Test Acc: 0.6327
Epoch 09 | Train Loss: 0.6926, Train Acc: 0.5198 | Test Loss: 0.6942, Test Acc: 0.5535
Epoch 10 | Train Loss: 0.6915, Train Acc: 0.5230 | Test Loss: 0.6946, Test Acc: 0.5230
Epoch 11 | Train Loss: 0.6910, Train Acc: 0.5282 | Test Loss: 0.6954, Test Acc: 0.5117
Epoch 12 | Train Loss: 0.6907, Train Acc: 0

In [33]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        preds = model(xb).argmax(1).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(yb.numpy())

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

print("Confusion Matrix:")
print(confusion_matrix(all_targets, all_preds))

print("\nClassification Report:")
print(classification_report(all_targets, all_preds, digits=3))


Confusion Matrix:
[[ 5994  6770]
 [13919 15721]]

Classification Report:
              precision    recall  f1-score   support

           0      0.301     0.470     0.367     12764
           1      0.699     0.530     0.603     29640

    accuracy                          0.512     42404
   macro avg      0.500     0.500     0.485     42404
weighted avg      0.579     0.512     0.532     42404



In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

lr = LogisticRegression(max_iter=3000, class_weight="balanced")
lr.fit(X_train, y_train)
pred = lr.predict(X_test)

print("LogReg accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, digits=3))


LogReg accuracy: 0.5014385435336289
              precision    recall  f1-score   support

           0      0.301     0.496     0.375     12764
           1      0.699     0.504     0.585     29640

    accuracy                          0.501     42404
   macro avg      0.500     0.500     0.480     42404
weighted avg      0.579     0.501     0.522     42404



In [35]:
print("y unique values and counts:")
print(y.value_counts())


y unique values and counts:
Crop_Health_Label
1    148198
0     63821
Name: count, dtype: int64
